In [16]:
#RUN QUESTO (1)

import os 
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [17]:
#RUN QUESTO (2)

# check GPU

import torch
print(torch.cuda.is_available())

True


In [18]:
#RUN QUESTO (3)

# Libraries

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from sklearn.decomposition import PCA
#from scipy.sparse import coo_matrix
from torch_geometric.data import HeteroData
from torch_geometric.utils import from_scipy_sparse_matrix
import torch_geometric.transforms as T
from torch_geometric.transforms import AddMetaPaths
from torch_geometric.nn import Linear, to_hetero
from torch_geometric.nn.conv import GATv2Conv
from sklearn.metrics import f1_score, roc_auc_score

fname_base = 'DyHANE/'
base = '2019-2021/' #'2019-2022/'
new = '2022/'

BASE MODEL

In [21]:
# load node features

AX = torch.load(fname_base+base+'features/AX_tensor.pt') 
AY = torch.load(fname_base+base+'features/AY_tensor.pt')
PX = torch.load(fname_base+base+'features/PX_tensor.pt')
IX = torch.load(fname_base+base+'features/IX_tensor.pt')

In [22]:
print(AX.shape)
print(PX.shape)
print(IX.shape)

torch.Size([13116, 389])
torch.Size([8641, 779])
torch.Size([1612, 489])


In [23]:
#Optional: same dimnsion for all node types

from sklearn.decomposition import PCA

dim=128 #32 #64

pca_author=PCA(n_components=dim).fit(AX)
pca_paper=PCA(n_components=dim).fit(PX)
pca_institution=PCA(n_components=dim).fit(IX)

In [24]:
# Load edges

AP = pd.read_csv(fname_base+base+'AP_map.csv')[['author', 'paper']] #new
#AP_coo = coo_matrix(AP.values.T)
AP_tensor = torch.tensor(AP.values.T)

PA = pd.read_csv(fname_base+base+'AP_map.csv')[['paper', 'author']]
#PA_coo = coo_matrix(PA.values.T)
PA_tensor = torch.tensor(PA.values.T)

PP = pd.read_csv(fname_base+base+'PP_map.csv')[['citing', 'cited']]
#PP_coo = coo_matrix(PP.values.T)
PP_tensor = torch.tensor(PP.values.T)

PP_rev = pd.read_csv(fname_base+base+'PP_map.csv')[['cited', 'citing']]
#PP_rev_coo = coo_matrix(PP_rev.values.T)
PP_rev_tensor = torch.tensor(PP_rev.values.T)

AI = pd.read_csv(fname_base+base+'AI_map.csv')
#AI_coo = coo_matrix(AI.values.T)
AI_tensor = torch.tensor(AI.values.T)

IA = AI[['institution_id', 'author_id']]
#IA_coo = coo_matrix(IA.values.T)
IA_tensor = torch.tensor(IA.values.T)

In [25]:
# Build HeteroData

data = HeteroData()

data['author'].x = torch.tensor(pca_author.transform(AX)).float()# qui
data['author'].y = AY
data['paper'].x = torch.tensor(pca_paper.transform(PX)).float()
data['institution'].x = torch.tensor(pca_institution.transform(IX)).float()

data['author', 'writes', 'paper'].edge_index = AP_tensor
data['paper', 'is_written_by', 'author'].edge_index = PA_tensor
data['paper', 'cites', 'paper'].edge_index = PP_tensor
data['paper', 'is_cited_by', 'paper'].edge_index = PP_rev_tensor
data['author', 'is_affiliated_with', 'institution'].edge_index = AI_tensor
data['institution', 'has_affiliated', 'author'].edge_index = IA_tensor

#data['paper', 'cites', 'paper'].edge_attr = ... # [num_edges_cites, num_features_cites]

#reversed edges explicitly added since ---data = T.ToUndirected()(data)--- is not working for edges between nodes of same type

print(data.node_types)
print(data.num_nodes)
print(data.edge_types)
print(data.num_edges)

#data = data.pin_memory()
#data = data#.to('cuda:0', non_blocking=True)

['author', 'paper', 'institution']
23369
[('author', 'writes', 'paper'), ('paper', 'is_written_by', 'author'), ('paper', 'cites', 'paper'), ('paper', 'is_cited_by', 'paper'), ('author', 'is_affiliated_with', 'institution'), ('institution', 'has_affiliated', 'author')]
40628


In [26]:
#Add metapaths

metapaths = [[('author', 'paper'), ('paper', 'author')], #APA
             [('author', 'paper'), ('paper', 'is_cited_by', 'paper'), ('paper', 'author')], #APPA
             [('author', 'institution'), ('institution', 'author')]] #AIA
data = AddMetaPaths(metapaths, weighted=True)(data)

In [27]:
data

HeteroData(
  metapath_dict={
    (author, metapath_0, author)=[2],
    (author, metapath_1, author)=[3],
    (author, metapath_2, author)=[2],
  },
  author={
    x=[13116, 128],
    y=[13116],
  },
  paper={ x=[8641, 128] },
  institution={ x=[1612, 128] },
  (author, writes, paper)={ edge_index=[2, 16020] },
  (paper, is_written_by, author)={ edge_index=[2, 16020] },
  (paper, cites, paper)={ edge_index=[2, 113] },
  (paper, is_cited_by, paper)={ edge_index=[2, 113] },
  (author, is_affiliated_with, institution)={ edge_index=[2, 4181] },
  (institution, has_affiliated, author)={ edge_index=[2, 4181] },
  (author, metapath_0, author)={
    edge_index=[2, 57736],
    edge_weight=[57736],
  },
  (author, metapath_1, author)={
    edge_index=[2, 302],
    edge_weight=[302],
  },
  (author, metapath_2, author)={
    edge_index=[2, 18507],
    edge_weight=[18507],
  }
)

In [12]:
print(data.has_isolated_nodes())
print(data.has_self_loops())
print(data.is_undirected())

#data.edge_types
#data['author', 'writes', 'paper'].edge_index
#print(torch.max(data['author', 'writes', 'paper'].edge_index))

#data.x_dict
#data.edge_index_dict
#data.metapath_dict


True
True
False


In [28]:
# Train - Val - Test split: random

import torch_geometric.transforms as T

#transform = T.compose([T.NormalizeFeatures(), T.RandomNodeSplit()])
transform = T.RandomNodeSplit()
data = transform(data)

In [12]:
print('Autori effettivi training:', data['author'].train_mask.count_nonzero())
print('Autori effettivi testing:', data['author'].test_mask.count_nonzero()) #testo su 1000 (500 val)

Autori effettivi training: tensor(11616)
Autori effettivi testing: tensor(1000)


In [19]:
#RUN QUESTO (4)

# Methods for statistics computation

from statistics import stdev

def process_metric(df, col):
    data = df[col]
    media = data.mean()
    st_dev = stdev(data.tolist())
    return media, st_dev

#df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
def processing_results(df):
    res = pd.DataFrame(columns=df.columns) #columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC']
    
    #Compute mean and standard deviation
    for col in df.columns:
        lista = df[col].tolist()
        media, st_dev = process_metric(df, col)
        stringa = "{:0.4f}".format(media)+u"\u00B1"+"{:0.4f}".format(st_dev)
        lista.append(stringa)
        res[col] = lista

    return res

In [20]:
#RUN QUESTO (5)

# GNN

class GAT(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, dropout=0):
        super().__init__()
        self.conv1 = GATv2Conv((-1, -1), hidden_channels, add_self_loops=False, dropout=dropout)
        self.lin1 = Linear(-1, hidden_channels)
        self.conv2 = GATv2Conv((-1, -1), out_channels, add_self_loops=False, dropout=dropout)
        self.lin2 = Linear(-1, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index) + self.lin1(x.relu()) #self.lin1(x) 
        x = x.relu()
        x = self.conv2(x, edge_index) + self.lin2(x.relu()) #self.lin2(x)
        return x

In [6]:
# Model training    

def train_node_classifier(model, data, optimizer, criterion, n_epochs=200): 

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x_dict, data.edge_index_dict)#['author'] #davide
        mask = data['author'].train_mask
        #loss = F.cross_entropy(out[mask], data['author'].y[mask])
        #loss = criterion(out[graph.train_mask], graph.y[graph.train_mask])
        loss = criterion(out['author'][mask], data['author'].y[mask]) #davide
        #loss = criterion(out[mask], data['author'].y[mask])
        loss.backward()
        optimizer.step()

        #pred = (out.x_dict, out.edge_index_dict).argmax(dim=-1)
        pred = out['author'].argmax(dim=1) ## Use the class with highest probability. #con davide
        #pred = out.argmax(dim=1)
        f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, data) #, auc

        if epoch % 20 == 0:
            print(f'Epoch: {epoch:03d}, Train Loss: {loss:.3f}, Val f1_micro: {f1_micro:.3f}')

    return model


def eval_node_classifier(model, data):

    model.eval()
    with torch.no_grad():
        #pred = model(data['author']).argmax(dim=1)
        #pred = model(data.x_dict, data.edge_index_dict).argmax(dim=-1) 
        pred = model(data.x_dict, data.edge_index_dict)['author'].argmax(dim=-1)
        pred_prob = torch.nn.functional.softmax(model(data.x_dict, data.edge_index_dict)['author'], -1)
        #print(pred_prob[0].sum())
        #pred_prob = model(data.x_dict, data.edge_index_dict)['author'] #prova per auc
        mask = data['author'].val_mask
        #acc = (pred[mask] == data['author'].y[mask]).sum() / mask.sum()
        correct = (pred[mask] == data['author'].y[mask]).sum()
        #acc = int(correct) / int(mask.sum())
        f1_micro = f1_score(data['author'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['author'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['author'].y.cpu(), pred.cpu(), average='weighted')
        #servono predicted probabilities, 8 classes
        auc = roc_auc_score(data['author'].y.cpu(), pred_prob.cpu().detach().numpy(), average='macro', multi_class='ovo')
        return f1_micro, f1_macro, f1_weigh, auc
  
  
num_run = 5
l_micro = []
l_macro = []
l_weigh = []
l_auc = []
for run in range(num_run):
    print('##### RUN '+str((run+1))+' #####')

    model = None
    model = GAT(hidden_channels=64, out_channels=8, dropout=0.4) #num_classes
    model = to_hetero(model, data.metadata(), aggr='sum')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001) #lr=0.005
    weights = torch.tensor([1.5, 1.5, 1, 1, 1, 1, 1.5, 1]).float().to(device)
    criterion = nn.CrossEntropyLoss(weights) #F.cross_entropy()

    model = train_node_classifier(model, data, optimizer, criterion, n_epochs=500) #500

    torch.save(model.state_dict(), fname_base+base+'base_model.pth')

    f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, data) #, auc
    #print(f'Test Eval: {test_ev:.3f}')
    print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, roc-auc: {auc:.3f}')

    l_micro.append(f1_micro)
    l_macro.append(f1_macro)
    l_weigh.append(f1_weigh)
    l_auc.append(auc)

df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
df['F1_micro'] = l_micro
df['F1_macro'] = l_macro
df['F1_weighted'] = l_weigh
df['ROC-AUC'] = l_auc
df_ok = processing_results(df)
df_ok.to_excel(fname_base+base+'results_base_model.xlsx', index=False)


##### RUN 1 #####


NameError: name 'data' is not defined

In [21]:
#RUN QUESTO (6)

# Model loading

def load_model(hidden_channels,out_channels,dropout,data,weigths_filename):
    model = GAT(hidden_channels, out_channels,dropout)
    model = to_hetero(model, data.metadata(), aggr='sum')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)

    with torch.no_grad():
        model.eval()
        model(data.x_dict, data.edge_index_dict)
        model.train()
    model.load_state_dict(torch.load(weigths_filename))
    return model

In [32]:
# Alternative to training: load saved model  

def eval_node_classifier(model, data):

    model.eval()
    with torch.no_grad(): 
        pred = model(data.x_dict, data.edge_index_dict)['author'].argmax(dim=-1)
        pred_prob = torch.nn.functional.softmax(model(data.x_dict, data.edge_index_dict)['author'], -1)
        mask = data['author'].val_mask
        correct = (pred[mask] == data['author'].y[mask]).sum()
        f1_micro = f1_score(data['author'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['author'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['author'].y.cpu(), pred.cpu(), average='weighted')
        #servono predicted probabilities, 8 classes
        auc = roc_auc_score(data['author'].y.cpu(), pred_prob.cpu().detach().numpy(), average='macro', multi_class='ovo')
        return f1_micro, f1_macro, f1_weigh, auc
  
model = GAT(hidden_channels=64, out_channels=8, dropout=0.4) #num_classes
model = to_hetero(model, data.metadata(), aggr='sum')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

##LOAD SAVED MODEL
model = load_model(64,8,0.4,data,fname_base+base+'model.pth')

f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, data) #, auc
#print(f'Test Eval: {test_ev:.3f}')
print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, , roc-auc: {auc:.3f}')



f1-micro: 0.644, f1-macro: 0.639, f1-weighted: 0.645, , roc-auc: 0.927


In [15]:
results = pd.read_excel(fname_base+base+'results_base_model.xlsx') 
print(results)


        F1_micro       F1_macro    F1_weighted        ROC-AUC
0       0.703644       0.701783       0.703872        0.94686
1       0.700442       0.696709       0.700083       0.945297
2       0.704483       0.701229       0.704601       0.946149
3       0.697088       0.693574       0.696932       0.944393
4       0.708676       0.705864        0.70914       0.947448
5  0.7029±0.0044  0.6998±0.0048  0.7029±0.0046  0.9460±0.0012


In [20]:
# Explainer

from torch_geometric.explain import Explainer, CaptumExplainer
explainer = Explainer(
    model=model,
    algorithm=CaptumExplainer('IntegratedGradients'), #InputXGradient
    explanation_type='phenomenon', #model's beahviour (model) vs individual predictions (phenomenon)
    node_mask_type='attributes', #attributes #'CaptumExplainer' only supports 'node_mask_type' None or 'attributes' (got 'object')
    edge_mask_type=None,
    model_config=dict(
        mode='multiclass_classification',
        task_level='node', 
        return_type='probs', #log_probs, raw
                                                                                                                                                 
    ),
)

#indices = torch.arange(13116)
#indices = torch.from_numpy(np.expand_dims(torch.arange(13116), axis=0))
with torch.no_grad():
    explanation = explainer(data.x_dict, data.edge_index_dict,target=data.y_dict["author"], index=torch.arange(13116)) #index=torch.arange(13116) #target=data.y if explanation_type=='phenomenon' else None
# print(f'Generated explanations in {explanation.available_explanations}')

# path = fname_base+'feature_importance.png'
# explanation.visualize_feature_importance(path, top_k=10)
# print(f"Feature importance plot has been saved to '{path}'")

# path = fname_base+'subgraph.pdf'
# explanation.visualize_graph(path)
# print(f"Subgraph visualization plot has been saved to '{path}'")

/root/.pyenv/versions/3.10.1/envs/DyHANE_cuda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
# B initialization

B = {}  
B_size = 768
topk = 64
n_types = ['author', 'paper', 'institution']
percs = [0] * len(n_types) 

#type importance, dato dalla media dei topk per ogni tipo
for i, nt in enumerate(n_types):
    values,indices=torch.topk(explanation.node_mask_dict[nt].sum(-1),k=topk)
    percs[i] = values.mean().item()

scaling_factor = B_size / sum(percs)
rates = [round(val * scaling_factor) for val in percs]

# top nodes for each type
for i, nt in enumerate(n_types):
    values,indices=torch.topk(explanation.node_mask_dict[nt].sum(-1),k=rates[i])
    B[nt+'s'] = indices.cpu().numpy().tolist()

with open(fname_base+base+'B_'+str(B_size)+'.pkl', 'wb') as fp:
    pickle.dump(B, fp)
    print('dictionary saved successfully to file')

print(B)


dictionary saved successfully to file
{'authors': [171, 65, 27, 1, 18, 9, 7468, 2, 1739, 205, 5243, 499, 5080, 340, 10168, 10059, 17, 12, 1095, 2821, 1464, 479, 454, 86, 93, 1377, 525, 303, 173, 7951, 2819, 189, 5043, 5466, 3218, 3310, 13006, 3087, 3264, 6733, 11211, 107, 109, 3252, 8586, 11884, 559, 3574, 5018, 11939, 7108, 1235, 2424, 10025, 396, 4738, 10710, 8294, 478, 11176, 661, 3370, 9221, 3120, 5995, 9779, 2848, 3369, 3517, 11848, 10908, 2658, 4469, 6817, 1998, 63, 66, 155, 11801, 502, 4331, 5919, 1159, 6544, 11382, 6331, 2885, 11410, 9873, 5877, 9719, 8293, 2255, 1576, 11320, 2441], 'papers': [6746, 5297, 3773, 5939, 6179, 4223, 5969, 2376, 4862, 5337, 2968, 5452, 5890, 4227, 2342, 5921, 6098, 5184, 4639, 131, 6378, 4971, 4939, 2692, 8042, 4715, 5089, 4564, 8457, 6100, 4199, 4940, 5125, 5565, 3077, 7964, 5102, 8066, 36, 5485, 5001, 3139, 4823, 6319, 30, 5360, 5729, 1821, 6377, 6110, 4019, 1975, 8132, 3420, 6125, 7649, 2205, 3768, 417, 578, 8181, 267, 1105, 1310, 3581, 5595, 394

INCREMENT

In [14]:
# I generation
# for each eij, I = I U {i, j} if category == 0 
# else I = I U {i, j} U N_i U N_j U N_mp_i if type(i) == 'A'

df_events = pd.read_csv(fname_base+new+'events_edges_cat.csv')
df_events.head(5)

,openalex_id_i,openalex_id_j,id_i,id_j,r_type,category
0,A5011101896,W1439977626,13116,8641,AP,1
1,A5077030086,W1439977626,13117,8641,AP,1
2,A5055416007,W2146875259,93,8642,AP,1
3,A5080395937,W2146875259,13118,8642,AP,1
4,A5037058917,W2146875259,13119,8642,AP,1


In [15]:
#mapped edges

#AP
AP_old = pd.read_csv(fname_base+base+'AP_map.csv')
AP_new = df_events[df_events['r_type'] == 'AP']
AP_new = AP_new[['id_i', 'id_j']]

#AI
AI_old = pd.read_csv(fname_base+base+'AI_map.csv')
AI_new = df_events[df_events['r_type'] == 'AI']
AI_new = AI_new[['id_i', 'id_j']]

#PP
PP_old = pd.read_csv(fname_base+base+'PP_map.csv')
PP_new = df_events[df_events['r_type'] == 'PP']
PP_new = PP_new[['id_i', 'id_j']]

#AA all
AA = pd.read_csv(fname_base+new+'AA_all_map.csv')

In [16]:
#serve il remapping degli edges. Per i nuovi ok, sono id_i e id_j di df_events.
#I vecchi li abbiamo già

def one_hop_neighborhood(category, id, ntype, AP_new, AP_old, AI_new, AI_old, PP_new, PP_old, AA): #all mapped
    nei_A = []
    nei_I = []
    nei_P = []

    if category == 1:
        if ntype == 'A':
            AP_old_sub = AP_old[AP_old['author'] == id]
            AP_new_sub = AP_new[AP_new['id_i'] == id]
            nei_P = list(set(AP_old_sub['paper'].tolist() + AP_new_sub['id_j'].tolist()))
            AI_old_sub = AI_old[AI_old['author_id'] == id]
            AI_new_sub = AI_new[AI_new['id_i'] == id]
            nei_I = list(set(AI_old_sub['institution_id'].tolist() + AI_new_sub['id_j'].tolist()))
            AA_sub = AA[AA['source'] == id]
            nei_A = AA_sub['target'].drop_duplicates().tolist()

        elif ntype == 'I':
            AI_old_sub = AI_old[AI_old['institution_id'] == id]
            AI_new_sub = AI_new[AI_new['id_j'] == id]
            nei_A = list(set(AI_old_sub['author_id'].tolist() + AI_new_sub['id_i'].tolist()))


        else: #ntype == 'P
            AP_old_sub = AP_old[AP_old['paper'] == id]
            AP_new_sub = AP_new[AP_new['id_j'] == id]
            nei_A = list(set(AP_old_sub['author'].tolist() + AP_new_sub['id_i'].tolist()))
            PP_old_sub = PP_old[PP_old['citing'] == id]
            PP_new_sub = PP_new[PP_new['id_i'] == id]
            nei_P = list(set(PP_old_sub['cited'].tolist() + PP_new_sub['id_j'].tolist()))
        
    return nei_A, nei_I, nei_P   

In [17]:
df_events['nei_A'],  df_events['nei_I'], df_events['nei_P'] = zip(*df_events.apply(lambda row: one_hop_neighborhood(row['category'], row['id_i'], row['r_type'][0], AP_new, AP_old, AI_new, AI_old, PP_new, PP_old, AA), axis=1))
df_events['nej_A'],  df_events['nej_I'], df_events['nej_P'] = zip(*df_events.apply(lambda row: one_hop_neighborhood(row['category'], row['id_j'], row['r_type'][1], AP_new, AP_old, AI_new, AI_old, PP_new, PP_old, AA), axis=1))
#df_events.head(5)


df_A = df_events[(df_events['r_type'] == 'AP') | (df_events['r_type'] == 'AI')]
A = df_A['id_i'].drop_duplicates().tolist()
A_nei_tmp = df_events['nei_A'].drop_duplicates().tolist() #list of lists
A_nei_tmp = [l for l in A_nei_tmp if l!=[]]
A_nei = [a for l in A_nei_tmp for a in l]
A_nej_tmp = df_events['nej_A'].drop_duplicates().tolist() #list of lists
A_nej_tmp = [l for l in A_nej_tmp if l!=[]]
A_nej = [a for l in A_nej_tmp for a in l]

A_all = list(set(A + A_nei + A_nej))

print(A_all[:10])


df_I = df_events[df_events['r_type'] == 'AI']
I = df_I['id_j'].drop_duplicates().tolist()
I_nei_tmp = df_events['nei_I'].drop_duplicates().tolist() #list of lists
I_nei_tmp = [l for l in I_nei_tmp if l!=[]]
I_nei = [i for l in I_nei_tmp for i in l]
I_nej_tmp = df_events['nej_I'].drop_duplicates().tolist() #list of lists
I_nej_tmp = [l for l in I_nej_tmp if l!=[]]
I_nej = [i for l in I_nej_tmp for i in l]

I_all = list(set(I + I_nei + I_nej))
print(I_all[:10])

#split AP and PP
df_P_fromAP = df_events[df_events['r_type'] == 'AP']
P_from_AP = df_P_fromAP['id_j'].drop_duplicates().tolist()
df_P_fromPP = df_events[df_events['r_type'] == 'PP']
P_from_PP = df_P_fromPP['id_i'].drop_duplicates().tolist() + df_P_fromPP['id_j'].drop_duplicates().tolist() 

#P = df_P['id_j'].drop_duplicates().tolist() if df_events['r_type'] == 'AP' else list(set(df_P['id_i'].drop_duplicates().tolist()+df_P['id_j'].drop_duplicates().tolist()))
P_nei_tmp = df_events['nei_P'].drop_duplicates().tolist() #list of lists
P_nei_tmp = [l for l in P_nei_tmp if l!=[]]
P_nei = [p for l in P_nei_tmp for p in l]
P_nej_tmp = df_events['nej_P'].drop_duplicates().tolist() #list of lists
P_nej_tmp = [l for l in P_nej_tmp if l!=[]]
P_nej = [p for l in P_nej_tmp for p in l]

P_all = list(set(P_from_AP + P_from_PP + P_nei + P_nej))
print(P_all[:10])

I = {}
I['authors'] = A_all
I['institutions'] = I_all
I['papers'] = P_all


with open(fname_base+new+'I.pkl', 'wb') as fp:
        pickle.dump(I, fp)
        print('dictionary saved successfully to file')

print(I)

[0, 3257, 7, 13, 15, 16, 21, 22, 23, 24]
[0, 3, 6, 12, 15, 17, 24, 27, 33, 34]
[4, 34, 35, 37, 41, 8244, 84, 91, 117, 122]
dictionary saved successfully to file
{'authors': [0, 3257, 7, 13, 15, 16, 21, 22, 23, 24, 8217, 26, 8219, 25, 33, 41, 8234, 46, 47, 8251, 63, 88, 93, 94, 99, 101, 107, 118, 127, 128, 132, 134, 135, 136, 8330, 8331, 139, 141, 142, 143, 8335, 145, 146, 140, 157, 159, 164, 8358, 169, 173, 179, 180, 181, 192, 4927, 4928, 223, 4929, 4930, 230, 231, 236, 240, 243, 244, 8435, 257, 265, 266, 4938, 8464, 273, 274, 275, 276, 281, 286, 300, 8494, 8502, 319, 8512, 321, 8511, 327, 330, 8528, 8529, 340, 352, 354, 355, 356, 369, 5876, 4962, 391, 4963, 6125, 411, 412, 413, 8611, 5878, 425, 426, 427, 428, 429, 430, 431, 450, 456, 457, 458, 6608, 484, 485, 486, 4985, 8704, 519, 6616, 8715, 527, 528, 534, 537, 539, 540, 541, 546, 6622, 557, 558, 559, 8754, 8755, 568, 582, 583, 595, 8788, 8790, 5005, 609, 8803, 8804, 8805, 8806, 8807, 619, 620, 640, 641, 657, 8852, 661, 6647, 6648, 8

In [18]:
print('Num authors in I:', len(I['authors']))
print('Num institutions in I:', len(I['institutions']))
print('Num papers in I:', len(I['papers']))

Num authors in I: 3409
Num institutions in I: 525
Num papers in I: 1817


In [6]:
##TEST DI PROVA

def indices_to_keep(I, B, ntype):
    if B == {}: #OnlineGNN
        return I[ntype]
    else: #continualGNN
        return list(set(I[ntype]+B[ntype]))
    
def build_testing_online_and_unchanged(AX,I,B):
    #prendiamo online
    indices_online = torch.load(fname_base+'testing_online.pt')
    mask_online = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_online[indices_online] = 1
    mask_online_opp = ~mask_online
    tx = torch.masked_select(torch.arange(AX.shape[0]), mask_online)
    p#rint('online shape',tx.shape)

    #Gli indici True di A sono quelli che si possono prendere, tra cui selezionare i random.
    #Quindi vanno messi a true gli indici non presenti in A e in B e non presenti in online.
    mask = torch.zeros(AX.shape[0], dtype=torch.bool)
    indices_no = indices_to_keep(I, B, 'authors')
    indices_si = [a for a in range(AX.shape[0]) if a not in indices_no] 
    mask[indices_si] = 1
    mask = torch.logical_and(mask, mask_online_opp)

    #prendiamo random 189 indici tra quelli true di A. 
    num_samples = indices_online.shape[0] #189
    true_indices = torch.masked_select(torch.arange(AX.shape[0]), mask) #tutti indici true 
    sampled_indices = true_indices[torch.randperm(true_indices.shape[0])[:num_samples]] #prendo num_samples valori a caso (sono gli indici che rimarrano True qui)
    mask_test = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_test[sampled_indices] = 1
    testing = torch.masked_select(torch.arange(AX.shape[0]), mask_test)
    #print('unchanged shape', testing.shape)

    mask_test = torch.logical_or(mask_test, mask_online) #metto a True indici che posso prendere & indici test online
    testing = torch.masked_select(torch.arange(AX.shape[0]), mask_test)
    #print('final shape', testing.shape)
    torch.save(testing, fname_base+'testing_online_and_unchanged.pt')


I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
B = pickle.load(open(fname_base+base+'B_256.pkl', 'rb'))
print('#########')
indices_online = torch.load(fname_base+'testing_online.pt')
print(indices_online.shape) #189 
AX = torch.load(fname_base+new+'features/AX_tensor_2019-2022.pt')  
build_testing_online_and_unchanged(AX, I, B)
print('#########')
indices_online_unchanged = torch.load(fname_base+'testing_online_and_unchanged.pt')
print(indices_online_unchanged.shape) #189*2




#########
torch.Size([189])
online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
#########
torch.Size([378])


In [22]:
#RUN QUESTO (7)

#Generate new heterodata 

def indices_to_keep(I, B, ntype):
    if B == {}: #OnlineGNN
        return I[ntype]
    else: #continualGNN
        return list(set(I[ntype]+B[ntype]))

def generate_mapped_edges(df_events_ok, r_type, I, B):
    df_r = df_events_ok[df_events_ok['r_type'] == r_type]
    df_r = df_events_ok[['id_i', 'id_j']]
    if r_type == 'AP':
        df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
    elif r_type == 'AI':
        df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
    else: #r_type == 'PP':
        df_r.rename(columns={'id_i': 'citing', 'id_j':'cited'}, inplace=True)
    return subset_mapped_edges(df_r, r_type, I, B)

def subset_mapped_edges(df_edg, r_type, I, B): #df_edg is AP_map or AI_map or PP_map
    col_i = df_edg.columns[0]
    col_j = df_edg.columns[1]
    i_type = r_type[0]
    j_type = r_type[1]
    diz = {'A': 'authors', 'P': 'papers', 'I': 'institutions'}
    if B == {}: #onlineGNN
        df_edg_sub = df_edg[(df_edg[col_i].isin(I[diz[i_type]])) & (df_edg[col_j].isin(I[diz[j_type]]))]
    else: #streamingGNN
        df_edg_sub = df_edg[((df_edg[col_i].isin(I[diz[i_type]])) | (df_edg[col_i].isin(B[diz[i_type]]))) & ((df_edg[col_j].isin(I[diz[j_type]])) | (df_edg[col_j].isin(B[diz[j_type]])))]
    return df_edg_sub


def build_testing_online_and_unchanged(AX,I,B):
    #prendiamo online
    indices_online = torch.load(fname_base+'testing_online.pt')
    mask_online = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_online[indices_online] = 1
    mask_online_opp = ~mask_online
    tx = torch.masked_select(torch.arange(AX.shape[0]), mask_online)
    #print('online shape',tx.shape)

    #Gli indici True di A sono quelli che si possono prendere, tra cui selezionare i random.
    #Quindi vanno messi a true gli indici non presenti in A e in B e non presenti in online.
    mask = torch.zeros(AX.shape[0], dtype=torch.bool)
    indices_no = indices_to_keep(I, B, 'authors')
    indices_si = [a for a in range(AX.shape[0]) if a not in indices_no] 
    mask[indices_si] = 1
    mask = torch.logical_and(mask, mask_online_opp)

    #prendiamo random 189 indici tra quelli true di A. 
    num_samples = indices_online.shape[0] #189
    true_indices = torch.masked_select(torch.arange(AX.shape[0]), mask) #tutti indici true 
    sampled_indices = true_indices[torch.randperm(true_indices.shape[0])[:num_samples]] #prendo num_samples valori a caso (sono gli indici che rimarrano True qui)
    mask_test = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_test[sampled_indices] = 1
    testing = torch.masked_select(torch.arange(AX.shape[0]), mask_test)
    #print('unchanged shape', testing.shape)

    mask_test = torch.logical_or(mask_test, mask_online) #metto a True indici che posso prendere & indici test online
    testing = torch.masked_select(torch.arange(AX.shape[0]), mask_test)
    #print('final shape', testing.shape)
    torch.save(testing, fname_base+'testing_online_and_unchanged.pt')
    


def generate_new_heterodata(I, B, df_events, scenario):
    for r_type in df_events['r_type'].drop_duplicates().tolist():
        df_ev = generate_mapped_edges(df_events, r_type, I, B)
        df_edg = pd.read_csv(fname_base+base+r_type+'_map.csv')
        df_base = subset_mapped_edges(df_edg, r_type, I, B)
        df_train = pd.concat([df_ev, df_base], axis=0).drop_duplicates()
        df_train.to_csv(fname_base+new+'Training/'+r_type+'_map.csv', index=False)

    #loading X and Y tensors
    AX = torch.load(fname_base+new+'features/AX_tensor_2019-2022.pt') 
    AY = torch.load(fname_base+new+'features/AY_tensor_2019-2022.pt')
    PX = torch.load(fname_base+new+'features/PX_tensor_2019-2022.pt')
    IX = torch.load(fname_base+new+'features/IX_tensor_2019-2022.pt')

    #PCA
    dim=128
    pca_author=PCA(n_components=dim).fit(AX)
    pca_paper=PCA(n_components=dim).fit(PX)
    pca_institution=PCA(n_components=dim).fit(IX)

    #create edge list
    AP = pd.read_csv(fname_base+new+'Training/AP_map.csv')[['author', 'paper']] 
    AP_tensor = torch.tensor(AP.values.T)
    PA = pd.read_csv(fname_base+new+'Training/AP_map.csv')[['paper', 'author']]
    PA_tensor = torch.tensor(PA.values.T)
    PP = pd.read_csv(fname_base+new+'Training/PP_map.csv')[['citing', 'cited']]
    PP_tensor = torch.tensor(PP.values.T)
    PP_rev = pd.read_csv(fname_base+new+'Training/PP_map.csv')[['cited', 'citing']]
    PP_rev_tensor = torch.tensor(PP_rev.values.T)
    AI = pd.read_csv(fname_base+new+'Training/AI_map.csv')
    AI_tensor = torch.tensor(AI.values.T)
    IA = AI[['institution_id', 'author_id']]
    IA_tensor = torch.tensor(IA.values.T)

    new_data = HeteroData()

    filter_nan=lambda x:x[:,torch.logical_not(x.isnan().long().sum(axis=0).bool())]

    new_data['author'].x = torch.tensor(pca_author.transform(AX)).float()# qui
    new_data['author'].y = AY.long()
    new_data['paper'].x = torch.tensor(pca_paper.transform(PX)).float()
    new_data['institution'].x = torch.tensor(pca_institution.transform(IX)).float()
    new_data['author', 'writes', 'paper'].edge_index = AP_tensor.long()
    new_data['paper', 'is_written_by', 'author'].edge_index = PA_tensor.long()
    new_data['paper', 'cites', 'paper'].edge_index = PP_tensor.long()
    new_data['paper', 'is_cited_by', 'paper'].edge_index = PP_rev_tensor.long()
    new_data['author', 'is_affiliated_with', 'institution'].edge_index = filter_nan(AI_tensor).long()
    new_data['institution', 'has_affiliated', 'author'].edge_index = filter_nan(IA_tensor).long()

    #add meta-paths
    metapaths = [[('author', 'paper'), ('paper', 'author')], #APA
            [('author', 'paper'), ('paper', 'is_cited_by', 'paper'), ('paper', 'author')], #APPA
            [('author', 'institution'), ('institution', 'author')]] #AIA
    new_data = AddMetaPaths(metapaths, weighted=True)(new_data)

    # transform: train-test split --> non lo facciamo perché abbiamo test set fissato
    #transform = T.RandomNodeSplit()
    #new_data = transform(new_data)


    mask_A = torch.zeros(AX.shape[0], dtype=torch.bool)
    indices_A = indices_to_keep(I, B, 'authors')
    mask_A[indices_A] = 1
    #caso random split:
    #new_data['author'].train_mask = torch.logical_and(new_data['author'].train_mask, mask_A) #T.IndexToMask(indices_A)
    #new_data['author'].val_mask = torch.logical_and(new_data['author'].val_mask, mask_A)
    #new_data['author'].test_mask = torch.logical_and(new_data['author'].test_mask, mask_A)
    
    #per avere stesso test altri modelli
    #prendi test set
    if scenario == 'C':
        indices_test = torch.load(fname_base+'testing_online.pt') #testing_online_and_unchanged.pt
    else:
        build_testing_online_and_unchanged(AX,I,B)
        indices_test = torch.load(fname_base+'testing_online_and_unchanged.pt')
    mask_test = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_test[indices_test] = 1
    mask_test_opp = ~mask_test

    new_data['author'].test_mask = torch.logical_and(mask_A, mask_test) #metto a True indici che posso prendere & indici test online
    new_data['author'].train_mask = torch.logical_and(mask_A, mask_test_opp) #metto a True indici che posso prendere & not indici test online
    new_data['author'].val_mask = torch.logical_and(mask_A, mask_test_opp) #metto a True indici che posso prendere & not indici test online
    #prendiamo random 500 indici tra quelli true di train e li spostiamo nel validation
    true_indices_val = torch.masked_select(torch.arange(AX.shape[0]), new_data['author'].val_mask) #tutti indici true di val
    sampled_indices = true_indices_val[torch.randperm(true_indices_val.shape[0])[:500]] #prendo 500 valori a caso (sono gli indici che rimarrano True qui e diventeranno False in train_mask)
    mask_val = torch.zeros(AX.shape[0], dtype=torch.bool)
    mask_val[sampled_indices] = 1
    new_data['author'].val_mask = torch.logical_and(new_data['author'].val_mask, mask_val)
    #quegli stessi 500 valori vanno messi a false per il train
    mask_train = ~mask_val
    new_data['author'].train_mask = torch.logical_and(new_data['author'].train_mask, mask_train)

    mask_I = torch.zeros(IX.shape[0], dtype=torch.bool)
    indices_I = indices_to_keep(I, B, 'institutions')
    mask_I[indices_I] = 1
    new_data['institution'].train_mask = mask_I
    
    mask_P = torch.zeros(PX.shape[0], dtype=torch.bool)
    indices_P = indices_to_keep(I, B, 'papers')
    mask_P[indices_P] = 1
    new_data['paper'].train_mask = mask_P
    
    return new_data

In [26]:
#INCREMENTAL STEP

I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
B = pickle.load(open(fname_base+base+'B_256.pkl', 'rb'))
#B = {}
df_events = pd.read_csv(fname_base+new+'events_edges.csv') #openalex_id_i, openalex_id_j, id_i, id_j, r_type
scenario = 'C' #'C+U'
new_data = generate_new_heterodata(I, B, df_events, scenario)

#print(new_data['author'].test_mask.count_nonzero()) #qua testo non su 1000 ma su 242 

num_run = 5
l_micro = []
l_macro = []
l_weigh = []
l_auc = []
for run in range(num_run):
    print('##### RUN '+str((run+1))+' #####')

    model = None
    model = GAT(hidden_channels=64, out_channels=8, dropout=0.4) #num_classes
    model = to_hetero(model, new_data.metadata(), aggr='sum')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    new_data, model = new_data.to(device), model.to(device)

    model = load_model(64,8,0.6, new_data,fname_base+base+'model.pth')

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.001) #0.01, 005, 001, 0001 fa cacare
    weights = torch.tensor([2, 2, 1, 1.5, 1.5, 1, 2, 1]).float().to(device)
    criterion = nn.CrossEntropyLoss(weights) 

    model = train_node_classifier(model, new_data, optimizer, criterion, n_epochs=500) #500

    torch.save(model.state_dict(), fname_base+new+'model_dyhane_B256_old_new.pth')

    f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, new_data) 
    #print(f'Test Eval: {test_ev:.3f}')
    print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, , roc-auc: {auc:.3f}')

    l_micro.append(f1_micro)
    l_macro.append(f1_macro)
    l_weigh.append(f1_weigh)
    l_auc.append(auc)

df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
df['F1_micro'] = l_micro
df['F1_macro'] = l_macro
df['F1_weighted'] = l_weigh
df['ROC-AUC'] = l_auc
df_ok = processing_results(df)
df_ok.to_excel(fname_base+new+'results_dyhane_B256_old_new.xlsx', index=False)

/tmp/ipykernel_843305/1398315810.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
/tmp/ipykernel_843305/1398315810.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
/tmp/ipykernel_843305/1398315810.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 

##### RUN 1 #####


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

Da qui in poi: UTILS

In [47]:
#Grid search

from itertools import product

# Example usage:
# Assuming you have a model called 'model' and hyperparameters defined as a dictionary
# hyperparameters = {'param1': [value1, value2, ...], 'param2': [value1, value2, ...], ...}

def train_node_classifier(model, data, optimizer, criterion, n_epochs=200): 

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x_dict, data.edge_index_dict)
        mask = data['author'].train_mask
        loss = criterion(out['author'][mask], data['author'].y[mask]) 
        loss.backward()
        optimizer.step()
        pred = out['author'].argmax(dim=1) ## Use the class with highest probability
        f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, data) 
        '''if epoch % 20 == 0:
            print(f'Epoch: {epoch:03d}, Train Loss: {loss:.3f}, Val f1_micro: {f1_micro:.3f}')'''
    return model


def eval_node_classifier(model, data):

    model.eval()
    with torch.no_grad(): 
        pred = model(data.x_dict, data.edge_index_dict)['author'].argmax(dim=-1)
        pred_prob = torch.nn.functional.softmax(model(data.x_dict, data.edge_index_dict)['author'], -1)
        mask = data['author'].val_mask
        correct = (pred[mask] == data['author'].y[mask]).sum()
        f1_micro = f1_score(data['author'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['author'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['author'].y.cpu(), pred.cpu(), average='weighted')
        #servono predicted probabilities, 8 classes
        auc = roc_auc_score(data['author'].y.cpu(), pred_prob.cpu().detach().numpy(), average='macro', multi_class='ovo')
        return f1_micro, f1_macro, f1_weigh, auc
    

def grid_search(hyperparameters):
    best_score = float('-inf')
    best_params = None
    
    # Generate all combinations of hyperparameters
    param_combinations = product(*hyperparameters.values())
    
    for params in param_combinations:
        print('########## Hyperparameters combination ##########')
        print(params)
        print('##########')
        I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
        B = pickle.load(open(fname_base+base+'B_'+str(params[0])+'.pkl', 'rb')) #params[0] --> 'B_size'
        df_events = pd.read_csv(fname_base+new+'events_edges.csv') #openalex_id_i, openalex_id_j, id_i, id_j, r_type
        scenario = 'C+U' #'C'
        new_data = generate_new_heterodata(I, B, df_events, scenario)

        num_run = 5
        l_micro = []
        l_macro = []
        l_weigh = []
        l_auc = []
        for run in range(num_run):
            print('##### RUN '+str((run+1))+' #####')

            model = None
            model = GAT(hidden_channels=64, out_channels=8, dropout=params[1]) #params[1] --> 'dropout'
            model = to_hetero(model, new_data.metadata(), aggr='sum')

            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            new_data, model = new_data.to(device), model.to(device)

            model = load_model(64,8,params[1], new_data,fname_base+base+'base_model.pth') #params[1] --> 'dropout'

            optimizer = torch.optim.Adam(model.parameters(), lr=params[2], weight_decay=params[3]) #params[2] --> lr, params[3] --> L2
            weights = torch.tensor([2, 2, 1, 1.5, 1.5, 1, 2, 1]).float().to(device)
            criterion = nn.CrossEntropyLoss(weights) 

            model = train_node_classifier(model, new_data, optimizer, criterion, n_epochs=500) 

            torch.save(model.state_dict(), fname_base+new+'model_dyhane_B'+str(params[0])+'_run'+str(run+1)+'_old_new.pth') #params[0] --> 'B_size'

            f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, new_data) 
            print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, , roc-auc: {auc:.3f}')

            l_micro.append(f1_micro)
            l_macro.append(f1_macro)
            l_weigh.append(f1_weigh)
            l_auc.append(auc)

        df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
        df['F1_micro'] = l_micro
        df['F1_macro'] = l_macro
        df['F1_weighted'] = l_weigh
        df['ROC-AUC'] = l_auc
        df_ok = processing_results(df)
        #df_ok.to_excel(fname_base+new+'results_dyhane_B'+str(params[0])+'_old_new.xlsx', index=False) #params[0] --> 'B_size'

        #score = df_ok['F1_micro'].mean() #F1 micro
        score = float(df_ok['F1_micro'].iloc[-1].split('±')[0])

        if score > best_score:
            best_score = score
            best_params = params

        
    
    return best_params, best_score



hyperparameters = {
    'B_size':[768, 2048, 256], #512 tutti e 80, 1024 ne mancano 7
    'dropout':[0.3, 0.4, 0.5, 0.6, 0.7],
    'lr': [0.001, 0.005, 0.01, 0.05],
    'L2': [0.0001, 0.0005, 0.001, 0.005]
}

best_params, best_score = grid_search(hyperparameters)
print('##########')
print("Best hyperparameters:", best_params)
print("Best validation score:", best_score)
print('##########')

########## Hyperparameters combination ##########
(512, 0.3, 0.001, 0.0001)
##########


/tmp/ipykernel_4879/1398315810.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'citin

online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
##### RUN 1 #####
Epoch: 020, Train Loss: 2.095, Val f1_micro: 0.193
Epoch: 040, Train Loss: 1.816, Val f1_micro: 0.223
Epoch: 060, Train Loss: 1.656, Val f1_micro: 0.236
Epoch: 080, Train Loss: 1.540, Val f1_micro: 0.241
Epoch: 100, Train Loss: 1.441, Val f1_micro: 0.250
Epoch: 120, Train Loss: 1.365, Val f1_micro: 0.256
Epoch: 140, Train Loss: 1.296, Val f1_micro: 0.259
Epoch: 160, Train Loss: 1.234, Val f1_micro: 0.263
Epoch: 180, Train Loss: 1.186, Val f1_micro: 0.272
Epoch: 200, Train Loss: 1.145, Val f1_micro: 0.272
Epoch: 220, Train Loss: 1.095, Val f1_micro: 0.279
Epoch: 240, Train Loss: 1.047, Val f1_micro: 0.279
Epoch: 260, Train Loss: 1.028, Val f1_micro: 0.282
Epoch: 280, Train Loss: 0.971, Val f1_micro: 0.289
Epoch: 300, Train Loss: 0.958, Val f1_micro: 0.288
Epoch: 320, Train Loss: 0.918, Val f1_micro: 0.291
Epoch: 340, Train Loss: 0.897, Val f1_micro: 0.294
Epoch: 360, Train Lo

/tmp/ipykernel_4879/1398315810.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'citin

online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
##### RUN 1 #####
Epoch: 020, Train Loss: 2.096, Val f1_micro: 0.189
Epoch: 040, Train Loss: 1.819, Val f1_micro: 0.218
Epoch: 060, Train Loss: 1.670, Val f1_micro: 0.230
Epoch: 080, Train Loss: 1.573, Val f1_micro: 0.236
Epoch: 100, Train Loss: 1.464, Val f1_micro: 0.246
Epoch: 120, Train Loss: 1.395, Val f1_micro: 0.251
Epoch: 140, Train Loss: 1.328, Val f1_micro: 0.256
Epoch: 160, Train Loss: 1.266, Val f1_micro: 0.260
Epoch: 180, Train Loss: 1.215, Val f1_micro: 0.264
Epoch: 200, Train Loss: 1.184, Val f1_micro: 0.268
Epoch: 220, Train Loss: 1.153, Val f1_micro: 0.272
Epoch: 240, Train Loss: 1.101, Val f1_micro: 0.271
Epoch: 260, Train Loss: 1.062, Val f1_micro: 0.275
Epoch: 280, Train Loss: 1.045, Val f1_micro: 0.276
Epoch: 300, Train Loss: 1.016, Val f1_micro: 0.277
Epoch: 320, Train Loss: 1.002, Val f1_micro: 0.279
Epoch: 340, Train Loss: 0.964, Val f1_micro: 0.281
Epoch: 360, Train Lo

...

online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
##### RUN 1 #####
Epoch: 020, Train Loss: 2.004, Val f1_micro: 0.191
Epoch: 040, Train Loss: 1.778, Val f1_micro: 0.215
Epoch: 060, Train Loss: 1.637, Val f1_micro: 0.224
Epoch: 080, Train Loss: 1.579, Val f1_micro: 0.230
Epoch: 100, Train Loss: 1.512, Val f1_micro: 0.236
Epoch: 120, Train Loss: 1.476, Val f1_micro: 0.239
Epoch: 140, Train Loss: 1.409, Val f1_micro: 0.240
Epoch: 160, Train Loss: 1.395, Val f1_micro: 0.234
Epoch: 180, Train Loss: 1.370, Val f1_micro: 0.241
Epoch: 200, Train Loss: 1.347, Val f1_micro: 0.247
Epoch: 220, Train Loss: 1.319, Val f1_micro: 0.245
Epoch: 240, Train Loss: 1.307, Val f1_micro: 0.254
Epoch: 260, Train Loss: 1.273, Val f1_micro: 0.249
Epoch: 280, Train Loss: 1.278, Val f1_micro: 0.249
Epoch: 300, Train Loss: 1.257, Val f1_micro: 0.256
Epoch: 320, Train Loss: 1.232, Val f1_micro: 0.260
Epoch: 340, Train Loss: 1.192, Val f1_micro: 0.255
Epoch: 360, Train Lo

/tmp/ipykernel_4879/1398315810.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'citin

online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
##### RUN 1 #####
Epoch: 020, Train Loss: 1.966, Val f1_micro: 0.202
Epoch: 040, Train Loss: 1.787, Val f1_micro: 0.204
Epoch: 060, Train Loss: 1.745, Val f1_micro: 0.211
Epoch: 080, Train Loss: 1.723, Val f1_micro: 0.220
Epoch: 100, Train Loss: 1.733, Val f1_micro: 0.211
Epoch: 120, Train Loss: 1.723, Val f1_micro: 0.218
Epoch: 140, Train Loss: 1.741, Val f1_micro: 0.213
Epoch: 160, Train Loss: 1.741, Val f1_micro: 0.209
Epoch: 180, Train Loss: 1.743, Val f1_micro: 0.217
Epoch: 200, Train Loss: 1.743, Val f1_micro: 0.222
Epoch: 220, Train Loss: 1.766, Val f1_micro: 0.222
Epoch: 240, Train Loss: 1.749, Val f1_micro: 0.213
Epoch: 260, Train Loss: 1.769, Val f1_micro: 0.215
Epoch: 280, Train Loss: 1.761, Val f1_micro: 0.213
Epoch: 300, Train Loss: 1.758, Val f1_micro: 0.211
Epoch: 320, Train Loss: 1.768, Val f1_micro: 0.202
Epoch: 340, Train Loss: 1.765, Val f1_micro: 0.209
Epoch: 360, Train Lo

/tmp/ipykernel_4879/1398315810.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author', 'id_j':'paper'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'author_id', 'id_j':'institution_id'}, inplace=True)
/tmp/ipykernel_4879/1398315810.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r.rename(columns={'id_i': 'citin

online shape torch.Size([189])
unchanged shape torch.Size([189])
final shape torch.Size([378])
##### RUN 1 #####
Epoch: 020, Train Loss: 1.924, Val f1_micro: 0.179
Epoch: 040, Train Loss: 1.684, Val f1_micro: 0.212
Epoch: 060, Train Loss: 1.563, Val f1_micro: 0.236
Epoch: 080, Train Loss: 1.413, Val f1_micro: 0.243
Epoch: 100, Train Loss: 1.315, Val f1_micro: 0.249
Epoch: 120, Train Loss: 1.246, Val f1_micro: 0.251
Epoch: 140, Train Loss: 1.162, Val f1_micro: 0.254
Epoch: 160, Train Loss: 1.084, Val f1_micro: 0.260
Epoch: 180, Train Loss: 1.015, Val f1_micro: 0.267
Epoch: 200, Train Loss: 0.964, Val f1_micro: 0.273
Epoch: 220, Train Loss: 0.897, Val f1_micro: 0.279
Epoch: 240, Train Loss: 0.845, Val f1_micro: 0.283
Epoch: 260, Train Loss: 0.781, Val f1_micro: 0.285
Epoch: 280, Train Loss: 0.770, Val f1_micro: 0.287
Epoch: 300, Train Loss: 0.719, Val f1_micro: 0.292
Epoch: 320, Train Loss: 0.679, Val f1_micro: 0.292
Epoch: 340, Train Loss: 0.610, Val f1_micro: 0.296
Epoch: 360, Train Lo

In [8]:
import pandas as pd
from itertools import product

#salva in DATA tutte le possibili combinazioni di parametri come df 
#(per ogni row una colonna con tupla combinazione)

hyperparameters = {
    'B_size':[768, 2048, 256], #512 tutti e 80, 1024 ne mancano 7
    'dropout':[0.3, 0.4, 0.5, 0.6, 0.7],
    'lr': [0.001, 0.005, 0.01, 0.05],
    'L2': [0.0001, 0.0005, 0.001, 0.005]
}

param_combinations = product(*hyperparameters.values())
param_conbinations_list = [params for params in param_combinations]
df = pd.DataFrame(columns=['Params'])
df['Params'] = param_conbinations_list
df.to_csv(fname_base+new+'params.csv', index=False)
df.head(5)


,Params
0,"(768, 0.3, 0.001, 0.0001)"
1,"(768, 0.3, 0.001, 0.0005)"
2,"(768, 0.3, 0.001, 0.001)"
3,"(768, 0.3, 0.001, 0.005)"
4,"(768, 0.3, 0.005, 0.0001)"


In [41]:
#estract combinations not processed
from itertools import product
import pandas as pd
fname_base = 'DyHANE/'

df1 = pd.read_csv(fname_base+'params_scores.csv')
print('Processed 1:', df1.shape[0])
df2 = pd.read_csv(fname_base+'params_scores_old.csv')
print('Processed 2:', df2.shape[0])
df2.rename(columns={'F1-micro':'F1_micro', 'F1-macro':'F1_macro', 'F1-weighted':'F1_weighted'}, inplace=True)
df = pd.concat([df1, df2], axis=0)#ignore_index=True
params_ok = df['Params'].tolist()
print('Already processed:',len(params_ok))
#print(params_ok[:10])


hyperparameters = {
    'B_size':[512, 2024], #768, 2048, 256
    'dropout':[0.3, 0.4, 0.5, 0.6, 0.7],
    'lr': [0.001, 0.005, 0.01, 0.05],
    'L2': [0.0001, 0.0005, 0.001, 0.005]
}

param_combinations = product(*hyperparameters.values())
param_combinations_list = [str(params) for params in param_combinations]
print('To check:',len(param_combinations_list))
print(param_combinations_list[:10])

#devono essere 7
params_to_process = [params for params in param_combinations_list if params not in params_ok]
print('To process',len(params_to_process))
print(params_to_process[:10])

To check: 160
['(512, 0.3, 0.001, 0.0001)', '(512, 0.3, 0.001, 0.0005)', '(512, 0.3, 0.001, 0.001)', '(512, 0.3, 0.001, 0.005)', '(512, 0.3, 0.005, 0.0001)', '(512, 0.3, 0.005, 0.0005)', '(512, 0.3, 0.005, 0.001)', '(512, 0.3, 0.005, 0.005)', '(512, 0.3, 0.01, 0.0001)', '(512, 0.3, 0.01, 0.0005)']
To process 80
['(2024, 0.3, 0.001, 0.0001)', '(2024, 0.3, 0.001, 0.0005)', '(2024, 0.3, 0.001, 0.001)', '(2024, 0.3, 0.001, 0.005)', '(2024, 0.3, 0.005, 0.0001)', '(2024, 0.3, 0.005, 0.0005)', '(2024, 0.3, 0.005, 0.001)', '(2024, 0.3, 0.005, 0.005)', '(2024, 0.3, 0.01, 0.0001)', '(2024, 0.3, 0.01, 0.0005)']


In [42]:
#check
import pandas as pd

def split_params(params):
    params = params[1:-1]
    params_list = params.split(', ')
    return params_list[0], params_list[1], params_list[2], params_list[3]

df1 = pd.read_csv(fname_base+'params_scores.csv')
#print('Processed 1:', df1.shape[0])
df2 = pd.read_csv(fname_base+'params_scores_old.csv')
#print('Processed 2:', df2.shape[0])
df2.rename(columns={'F1-micro':'F1_micro', 'F1-macro':'F1_macro', 'F1-weighted':'F1_weighted'}, inplace=True)
df = pd.concat([df1, df2], axis=0)#ignore_index=True

new_columns = df2['Params'].apply(split_params)
new_column_names = ['B_size', 'dropout', 'lr', 'L2']

for i, col_name in enumerate(new_column_names):
    df[col_name] = [res[i] for res in new_columns]

B_size = '1024' #153, '512' invece 0

# Count the number of rows matching the specified value for the specified column
count = (df2['B_size'] == B_size).sum()
print(count)

ValueError: Length of values (152) does not match length of index (392)

In [25]:
#RUN QUESTO (8)

from tqdm import tqdm
import pandas as pd


PATH_SAVE=fname_base+'params_scores_others.csv' #params_scores.csv


path=fname_base+new+'params_others.csv' #params.csv
DATA=pd.read_csv(path) 

DATA=DATA.to_dict(orient="records")

In [26]:
#RUN QUESTO (9)

from itertools import product

# Example usage:
# Assuming you have a model called 'model' and hyperparameters defined as a dictionary
# hyperparameters = {'param1': [value1, value2, ...], 'param2': [value1, value2, ...], ...}

def train_node_classifier(model, data, optimizer, criterion, n_epochs=200): 

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x_dict, data.edge_index_dict)
        mask = data['author'].train_mask
        loss = criterion(out['author'][mask], data['author'].y[mask]) 
        loss.backward()
        optimizer.step()
        pred = out['author'].argmax(dim=1) ## Use the class with highest probability
        f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, data) 
        '''if epoch % 20 == 0:
            print(f'Epoch: {epoch:03d}, Train Loss: {loss:.3f}, Val f1_micro: {f1_micro:.3f}')'''
    return model


def eval_node_classifier(model, data):

    model.eval()
    with torch.no_grad(): 
        pred = model(data.x_dict, data.edge_index_dict)['author'].argmax(dim=-1)
        pred_prob = torch.nn.functional.softmax(model(data.x_dict, data.edge_index_dict)['author'], -1)
        mask = data['author'].val_mask
        correct = (pred[mask] == data['author'].y[mask]).sum()
        f1_micro = f1_score(data['author'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['author'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['author'].y.cpu(), pred.cpu(), average='weighted')
        #servono predicted probabilities, 8 classes
        auc = roc_auc_score(data['author'].y.cpu(), pred_prob.cpu().detach().numpy(), average='macro', multi_class='ovo')
        return f1_micro, f1_macro, f1_weigh, auc
    

def grid_search(params):
    
    print('########## Hyperparameters combination ##########')
    print(params)
    print('##########')
    params = params[1:-1]
    I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
    B = pickle.load(open(fname_base+base+'B_'+str(params.split(', ')[0])+'.pkl', 'rb')) #params[0] --> 'B_size'
    df_events = pd.read_csv(fname_base+new+'events_edges.csv') #openalex_id_i, openalex_id_j, id_i, id_j, r_type
    scenario = 'C+U' #'C'
    new_data = generate_new_heterodata(I, B, df_events, scenario)

    num_run = 5
    l_micro = []
    l_macro = []
    l_weigh = []
    l_auc = []
    for run in range(num_run):
        #print('##### RUN '+str((run+1))+' #####')

        model = None
        model = GAT(hidden_channels=64, out_channels=8, dropout=float(params.split(', ')[1])) #params[1] --> 'dropout'
        model = to_hetero(model, new_data.metadata(), aggr='sum')

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        new_data, model = new_data.to(device), model.to(device)

        model = load_model(64,8,float(params.split(', ')[1]), new_data,fname_base+base+'base_model.pth') #params[1] --> 'dropout'

        optimizer = torch.optim.Adam(model.parameters(), lr=float(params.split(', ')[2]), weight_decay=float(params.split(', ')[3])) #params[2] --> lr, params[3] --> L2
        weights = torch.tensor([2, 2, 1, 1.5, 1.5, 1, 2, 1]).float().to(device)
        criterion = nn.CrossEntropyLoss(weights) 

        model = train_node_classifier(model, new_data, optimizer, criterion, n_epochs=500) 

        torch.save(model.state_dict(), fname_base+new+'model_dyhane_B'+str(params.split(', ')[0])+'_run'+str(run+1)+'_old_new.pth') #params[0] --> 'B_size'

        f1_micro, f1_macro, f1_weigh, auc = eval_node_classifier(model, new_data) 
        #print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, , roc-auc: {auc:.3f}')

        l_micro.append(f1_micro)
        l_macro.append(f1_macro)
        l_weigh.append(f1_weigh)
        l_auc.append(auc)

    df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
    df['F1_micro'] = l_micro
    df['F1_macro'] = l_macro
    df['F1_weighted'] = l_weigh
    df['ROC-AUC'] = l_auc
    df_ok = processing_results(df)
    #df_ok.to_excel(fname_base+new+'results_dyhane_B'+str(params[0])+'_old_new.xlsx', index=False) #params[0] --> 'B_size'

    #score = df_ok['F1_micro'].mean() #F1 micro
    f1_micro_tot = df_ok['F1_micro'].iloc[-1] #float(df_ok['F1_micro'].iloc[-1].split('±')[0])
    f1_macro_tot = df_ok['F1_macro'].iloc[-1]
    f1_weigh_tot = df_ok['F1_weighted'].iloc[-1]
    roc_auc_tot = df_ok['ROC-AUC'].iloc[-1]

    #print(f'f1-micro: {f1_micro_tot:.3f}, f1-macro: {f1_macro_tot:.3f}, f1-weighted: {f1_weigh_tot:.3f}, , roc-auc: {roc_auc_tot:.3f}')
    print('f1-micro:', f1_micro_tot, 'f1-macro:', f1_macro_tot, 'f1-weighted:', f1_weigh_tot, 'roc-auc:', roc_auc_tot)

    return [f1_micro_tot, f1_macro_tot, f1_weigh_tot, roc_auc_tot]




In [27]:
#RUN QUESTO (10)

pd.set_option('mode.chained_assignment', None)

if os.path.exists(PATH_SAVE):
    NEW_DATA=pd.read_csv(PATH_SAVE).to_dict(orient="records")
else:
    NEW_DATA=[]

for row in tqdm(DATA[len(NEW_DATA):]):

    params = row['Params']
    scores = grid_search(params)

    record={
        "Params":params,
        "F1_micro":scores[0],
        "F1_macro":scores[1],
        "F1_weighted":scores[2],
        "ROC-AUC":scores[3]
    }
    NEW_DATA.append(record)
    if len(NEW_DATA)%5==0:
        pd.DataFrame().from_records(NEW_DATA).to_csv(PATH_SAVE,index=False)

pd.DataFrame().from_records(NEW_DATA).to_csv(PATH_SAVE,index=False)

  0%|          | 0/8 [00:00<?, ?it/s]

########## Hyperparameters combination ##########
(1024, 0.7, 0.01, 0.0001)
##########


 12%|█▎        | 1/8 [05:57<41:43, 357.61s/it]

f1-micro: 0.3031±0.0013 f1-macro: 0.2977±0.0015 f1-weighted: 0.3027±0.0013 roc-auc: 0.6691±0.0015
########## Hyperparameters combination ##########
(1024, 0.7, 0.01, 0.0005)
##########


 25%|██▌       | 2/8 [11:59<35:59, 359.99s/it]

...

f1-micro: 0.2182±0.0070 f1-macro: 0.2054±0.0076 f1-weighted: 0.2029±0.0083 roc-auc: 0.6309±0.0020
########## Hyperparameters combination ##########
(1024, 0.7, 0.05, 0.001)
##########


 88%|████████▊ | 7/8 [42:00<06:00, 360.45s/it]

f1-micro: 0.2078±0.0033 f1-macro: 0.1925±0.0050 f1-weighted: 0.1901±0.0059 roc-auc: 0.6281±0.0026
########## Hyperparameters combination ##########
(1024, 0.7, 0.05, 0.005)
##########


100%|██████████| 8/8 [48:11<00:00, 361.44s/it]

f1-micro: 0.1797±0.0065 f1-macro: 0.1352±0.0119 f1-weighted: 0.1309±0.0113 roc-auc: 0.6202±0.0018


In [7]:
#Stupid test to check masking

import random

mask_A = torch.zeros(10, dtype=torch.bool)
indices_A = [0, 3, 4, 5, 8]
mask_A[indices_A] = 1
print('mask A:', mask_A)
   
indices_test = torch.tensor([0, 1, 2, 3])
mask_test = torch.zeros(10, dtype=torch.bool)
mask_test[indices_test] = 1
print('mask test:', mask_test)
mask_test_opp = ~mask_test
print('opposite of mask test:', mask_test_opp)

init_test_mask = torch.logical_and(mask_A, mask_test)
print('final test mask:', init_test_mask)
init_train_mask = torch.logical_and(mask_A, mask_test_opp)
print('init train mask:', init_train_mask)
init_val_mask = torch.logical_and(mask_A, mask_test_opp)
print('init val mask:', init_val_mask)

true_indices_val = torch.masked_select(torch.arange(10), init_val_mask) #tutti indici true di val
sampled_indices = true_indices_val[torch.randperm(true_indices_val.shape[0])[:2]] #prendo 500 valori a caso (sono gli indici che rimarrano True qui e diventeranno False in train_mask)
mask_val = torch.zeros(10, dtype=torch.bool)
mask_val[sampled_indices] = 1
val_mask = torch.logical_and(init_val_mask, mask_val)
print('final val mask:', val_mask)
mask_train = ~mask_val
train_mask = torch.logical_and(init_train_mask, mask_train)
print('final train mask:', train_mask)


mask A: tensor([ True, False, False,  True,  True,  True, False, False,  True, False])
mask test: tensor([ True,  True,  True,  True, False, False, False, False, False, False])
opposite of mask test: tensor([False, False, False, False,  True,  True,  True,  True,  True,  True])
final test mask: tensor([ True, False, False,  True, False, False, False, False, False, False])
init train mask: tensor([False, False, False, False,  True,  True, False, False,  True, False])
init val mask: tensor([False, False, False, False,  True,  True, False, False,  True, False])
final val mask: tensor([False, False, False, False, False,  True, False, False,  True, False])
final train mask: tensor([False, False, False, False,  True, False, False, False, False, False])


In [31]:
#CASO ONLINE: solo edges in new e risp. nodes


#loading X and Y tensors
AX = torch.load(fname_base+new+'features/AX_tensor_2019-2022.pt') 
AY = torch.load(fname_base+new+'features/AY_tensor_2019-2022.pt')
PX = torch.load(fname_base+new+'features/PX_tensor_2019-2022.pt')
IX = torch.load(fname_base+new+'features/IX_tensor_2019-2022.pt')

#PCA
dim=128
pca_author=PCA(n_components=dim).fit(AX)
pca_paper=PCA(n_components=dim).fit(PX)
pca_institution=PCA(n_components=dim).fit(IX)

#create edge list
AP = pd.read_csv(fname_base+new+'AP_map.csv')[['author', 'paper']] 
AP_tensor = torch.tensor(AP.values.T)
PA = pd.read_csv(fname_base+new+'AP_map.csv')[['paper', 'author']]
PA_tensor = torch.tensor(PA.values.T)
PP = pd.read_csv(fname_base+new+'PP_map.csv')[['citing', 'cited']]
PP_tensor = torch.tensor(PP.values.T)
PP_rev = pd.read_csv(fname_base+new+'PP_map.csv')[['cited', 'citing']]
PP_rev_tensor = torch.tensor(PP_rev.values.T)
AI = pd.read_csv(fname_base+new+'AI_map.csv')
AI_tensor = torch.tensor(AI.values.T)
IA = AI[['institution_id', 'author_id']]
IA_tensor = torch.tensor(IA.values.T)

new_data = HeteroData()

filter_nan=lambda x:x[:,torch.logical_not(x.isnan().long().sum(axis=0).bool())]

new_data['author'].x = torch.tensor(pca_author.transform(AX)).float()# qui
new_data['author'].y = AY.long()
new_data['paper'].x = torch.tensor(pca_paper.transform(PX)).float()
new_data['institution'].x = torch.tensor(pca_institution.transform(IX)).float()
new_data['author', 'writes', 'paper'].edge_index = AP_tensor.long()
new_data['paper', 'is_written_by', 'author'].edge_index = PA_tensor.long()
new_data['paper', 'cites', 'paper'].edge_index = PP_tensor.long()
new_data['paper', 'is_cited_by', 'paper'].edge_index = PP_rev_tensor.long()
new_data['author', 'is_affiliated_with', 'institution'].edge_index = filter_nan(AI_tensor).long()
new_data['institution', 'has_affiliated', 'author'].edge_index = filter_nan(IA_tensor).long()

#add meta-paths
metapaths = [[('author', 'paper'), ('paper', 'author')], #APA
        [('author', 'paper'), ('paper', 'is_cited_by', 'paper'), ('paper', 'author')], #APPA
        [('author', 'institution'), ('institution', 'author')]] #AIA
new_data = AddMetaPaths(metapaths, weighted=True)(new_data)

# transform: train-test split + index to mask
transform = T.RandomNodeSplit()
new_data = transform(new_data)

mask_A = torch.zeros(AX.shape[0], dtype=torch.bool)
#sono gli indici degli A nei nuovi AP + nei nuovi AI. Che poi sarebbe solo la differenza degli AX
indices_A = list(set(AP['author'].tolist()+AI['author_id'].tolist())) #indices_to_keep(I, B, 'authors') 
mask_A[indices_A] = 1
new_data['author'].train_mask = torch.logical_and(new_data['author'].train_mask, mask_A) #T.IndexToMask(indices_A)
new_data['author'].val_mask = torch.logical_and(new_data['author'].val_mask, mask_A)
new_data['author'].test_mask = torch.logical_and(new_data['author'].test_mask, mask_A)

mask_I = torch.zeros(IX.shape[0], dtype=torch.bool)
#sono gli indici degli I nei nuovi AI
indices_I = AI['institution_id'].drop_duplicates().tolist()#indices_to_keep(I, B, 'institutions') 
mask_I[indices_I] = 1
new_data['institution'].train_mask = mask_I

mask_P = torch.zeros(PX.shape[0], dtype=torch.bool)
#sono gli indici dei P nei nuovi AP + nei nuovi PP
indices_P = list(set(AP['paper'].tolist()+PP['citing'].tolist()+PP['cited'].tolist()))#indices_to_keep(I, B, 'papers') 
mask_P[indices_P] = 1
new_data['paper'].train_mask = mask_P

print(new_data)





HeteroData(
  metapath_dict={
    (author, metapath_0, author)=[2],
    (author, metapath_1, author)=[3],
    (author, metapath_2, author)=[2],
  },
  author={
    x=[15397, 128],
    y=[15397],
    train_mask=[15397],
    val_mask=[15397],
    test_mask=[15397],
  },
  paper={
    x=[10173, 128],
    train_mask=[10173],
  },
  institution={
    x=[1830, 128],
    train_mask=[1830],
  },
  (author, writes, paper)={ edge_index=[2, 2748] },
  (paper, is_written_by, author)={ edge_index=[2, 2748] },
  (paper, cites, paper)={ edge_index=[2, 79] },
  (paper, is_cited_by, paper)={ edge_index=[2, 79] },
  (author, is_affiliated_with, institution)={ edge_index=[2, 712] },
  (institution, has_affiliated, author)={ edge_index=[2, 712] },
  (author, metapath_0, author)={
    edge_index=[2, 8472],
    edge_weight=[8472],
  },
  (author, metapath_1, author)={
    edge_index=[2, 122],
    edge_weight=[122],
  },
  (author, metapath_2, author)={
    edge_index=[2, 1594],
    edge_weight=[1594],
  }
)

In [35]:
#keep indices of nodes used in test set by online model (mask values=False)

print(AX.shape[0])
new_data['author'].test_mask.count_nonzero() #mask tutti tranni 186 
#prendi gli indici in corrispondenza di false
training_online = torch.masked_select(torch.arange(AX.shape[0]), new_data['author'].test_mask)
print(training_online.shape)
torch.save(training_online, fname_base+'testing_online.pt')

15397
torch.Size([189])


In [67]:
#test set con nodi del test set dell'online model + un numero uguale di nodi unchanged non presenti né in I né in B
#se non presenti in I+B a maggior ragione non presenti nel test set di online

#prendiamo online
indices_online = torch.load(fname_base+'testing_online.pt')
mask_online = torch.zeros(AX.shape[0], dtype=torch.bool)
mask_online[indices_online] = 1
mask_online_opp = ~mask_online
tx = torch.masked_select(torch.arange(AX.shape[0]), mask_online)
print(tx.shape)
   

I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
B = pickle.load(open(fname_base+base+'B_512.pkl', 'rb'))

#Gli indici True di A sono quelli che si possono prendere, tra cui selezionare i random.
#Quindi vanno messi a true gli indici non presenti in A e in B e non presenti in online.
mask = torch.zeros(AX.shape[0], dtype=torch.bool)
indices_no = list(set(I['authors']+B['authors']))
indices_si = [a for a in range(AX.shape[0]) if a not in indices_no] 
mask[indices_si] = 1
mask = torch.logical_and(mask, mask_online_opp)

#prendiamo random 189 indici tra quelli true di A. 
true_indices_val = torch.masked_select(torch.arange(AX.shape[0]), mask) #tutti indici true di val
sampled_indices = true_indices_val[torch.randperm(true_indices_val.shape[0])[:189]] #prendo 189 valori a caso (sono gli indici che rimarrano True qui)

mask_test_2 = torch.zeros(AX.shape[0], dtype=torch.bool)
mask_test_2[sampled_indices] = 1
testing_2 = torch.masked_select(torch.arange(AX.shape[0]), mask_test_2)
print(testing_2.shape)

mask_test_2 = torch.logical_or(mask_test_2, mask_online) #metto a True indici che posso prendere & indici test online
testing_2 = torch.masked_select(torch.arange(AX.shape[0]), mask_test_2)
print(testing_2.shape)
torch.save(testing_2, fname_base+'testing_online_and_unchanged.pt')

torch.Size([189])
torch.Size([189])
torch.Size([378])


In [39]:
#ONLINE CON SUO TEST

res = pd.read_excel(fname_base+new+'results_online.xlsx')
res

,F1_micro,F1_macro,F1_weighted,ROC-AUC
0,0.280184,0.274059,0.277085,0.672901
1,0.283367,0.275166,0.279129,0.672345
2,0.286095,0.278059,0.2814,0.675729
3,0.288693,0.279408,0.283072,0.673634
4,0.288303,0.279634,0.283667,0.678202
5,0.2853±0.0036,0.2773±0.0025,0.2809±0.0028,0.6746±0.0024


In [64]:
print('Autori eff. training:', new_data['author'].train_mask.count_nonzero())
print('Papers eff. training:', new_data['paper'].train_mask.count_nonzero())
print('Institutions eff. training:', new_data['institution'].train_mask.count_nonzero())

Autori eff. training: tensor(3164, device='cuda:0')
Papers eff. training: tensor(2552, device='cuda:0')
Institutions eff. training: tensor(596, device='cuda:0')


In [61]:
print(new_data['author'].train_mask.count_nonzero())
print(new_data['author'].num_nodes)

print(new_data['paper'].train_mask.count_nonzero())
print(new_data['paper'].num_nodes)

tensor(4357, device='cuda:0')
15397
tensor(2940, device='cuda:0')
10173


In [ ]:
#check num nodi

AP_base = pd.read_csv(fname_base+base+'AP_map.csv')
print('A 2019-2021:', AP_base['author'].drop_duplicates().shape[0])

AP_new = pd.read_csv(fname_base+new+'AP_map.csv')
print('A 2022:', AP_new['author'].drop_duplicates().shape[0])

authors_base = AP_base['author'].drop_duplicates().tolist()
AP_x = AP_new[~AP_new['author'].isin(authors_base)]
print('A 2022 not in 2019-2021:', AP_x['author'].drop_duplicates().shape[0])

In [ ]:
I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
authors_I = I['authors']

print(len(authors_I))

x = [a for a in authors_I if a not in AP_x['author'].tolist()]
print(len(x))

x2 = [a for a in authors_I if a in AP_x['author'].tolist()]
print(len(x2))

In [1]:
#estract combinations not processed
from itertools import product
import pandas as pd
fname_base = 'DyHANE/'

df1 = pd.read_csv(fname_base+'params_scores.csv')
df2 = pd.read_csv(fname_base+'params_scores_old.csv')
df2.rename(columns={'F1-micro':'F1_micro', 'F1-macro':'F1_macro', 'F1-weighted':'F1_weighted'}, inplace=True)
df = pd.concat([df1, df2], ignore_index=True)
params_ok = df['Params'].tolist()

hyperparameters = {
    'B_size':[512, 2024], #768, 2048, 256
    'dropout':[0.3, 0.4, 0.5, 0.6, 0.7],
    'lr': [0.001, 0.005, 0.01, 0.05],
    'L2': [0.0001, 0.0005, 0.001, 0.005]
}

param_combinations = product(*hyperparameters.values())

params_to_process = [params for params in param_combinations if params not in params_ok]
print(len(params_to_process))

160


In [10]:
import pandas as pd
fname_base='DyHANE/'

df = pd.read_csv(fname_base+'params_scores_all.csv')

#mi serve il df ordinato per valori decrescenti di una metrica

def extract_score(value):
    return float(value.split('±')[0])

df['score'] = df.apply(lambda x: extract_score(x['ROC-AUC']), axis=1)
df_sorted = df.sort_values(by='score', ascending=False)
df_sorted.to_csv(fname_base+'params_scores_all.csv', index=False)
df_sorted.head(20)


,Params,F1_micro,F1_macro,F1_weighted,ROC-AUC,score
248,"(2048, 0.3, 0.01, 0.0001)",0.6818±0.0021,0.6766±0.0018,0.6823±0.0017,0.9561±0.0009,0.9561
244,"(2048, 0.3, 0.005, 0.0001)",0.6792±0.0016,0.6739±0.0014,0.6793±0.0015,0.9525±0.0010,0.9525
264,"(2048, 0.4, 0.01, 0.0001)",0.6809±0.0020,0.6756±0.0021,0.6813±0.0019,0.9512±0.0020,0.9512
260,"(2048, 0.4, 0.005, 0.0001)",0.6770±0.0014,0.6715±0.0012,0.6775±0.0011,0.9510±0.0005,0.9510
245,"(2048, 0.3, 0.005, 0.0005)",0.6678±0.0008,0.6624±0.0008,0.6672±0.0010,0.9509±0.0008,0.9509
84,"(1024, 0.3, 0.005, 0.0001)",0.6734±0.0024,0.6680±0.0023,0.6732±0.0020,0.9484±0.0011,0.9484
240,"(2048, 0.3, 0.001, 0.0001)",0.6460±0.0014,0.6401±0.0014,0.6432±0.0015,0.9477±0.0004,0.9477
276,"(2048, 0.5, 0.005, 0.0001)",0.6722±0.0012,0.6680±0.0011,0.6732±0.0009,0.9468±0.0004,0.9468
249,"(2048, 0.3, 0.01, 0.0005)",0.6738±0.0042,0.6678±0.0033,0.6736±0.0033,0.9465±0.0010,0.9465
168,"(768, 0.3, 0.01, 0.0001)",0.6739±0.0014,0.6684±0.0009,0.6746±0.0011,0.9462±0.0003,0.9462


In [11]:
#Analisi dei primi 30
df = pd.read_csv(fname_base+'params_scores_all.csv')
df = df[df['score'] >= 0.94] #32 combinazioni
df


,Params,F1_micro,F1_macro,F1_weighted,ROC-AUC,score
0,"(2048, 0.3, 0.01, 0.0001)",0.6818±0.0021,0.6766±0.0018,0.6823±0.0017,0.9561±0.0009,0.9561
1,"(2048, 0.3, 0.005, 0.0001)",0.6792±0.0016,0.6739±0.0014,0.6793±0.0015,0.9525±0.0010,0.9525
2,"(2048, 0.4, 0.01, 0.0001)",0.6809±0.0020,0.6756±0.0021,0.6813±0.0019,0.9512±0.0020,0.9512
3,"(2048, 0.4, 0.005, 0.0001)",0.6770±0.0014,0.6715±0.0012,0.6775±0.0011,0.9510±0.0005,0.9510
4,"(2048, 0.3, 0.005, 0.0005)",0.6678±0.0008,0.6624±0.0008,0.6672±0.0010,0.9509±0.0008,0.9509
5,"(1024, 0.3, 0.005, 0.0001)",0.6734±0.0024,0.6680±0.0023,0.6732±0.0020,0.9484±0.0011,0.9484
6,"(2048, 0.3, 0.001, 0.0001)",0.6460±0.0014,0.6401±0.0014,0.6432±0.0015,0.9477±0.0004,0.9477
7,"(2048, 0.5, 0.005, 0.0001)",0.6722±0.0012,0.6680±0.0011,0.6732±0.0009,0.9468±0.0004,0.9468
8,"(2048, 0.3, 0.01, 0.0005)",0.6738±0.0042,0.6678±0.0033,0.6736±0.0033,0.9465±0.0010,0.9465
9,"(768, 0.3, 0.01, 0.0001)",0.6739±0.0014,0.6684±0.0009,0.6746±0.0011,0.9462±0.0003,0.9462


In [13]:
import pickle
I = pickle.load(open('DyHANE/2022/I.pkl', 'rb'))
print(len(I['authors']))
print(len(I['papers']))
print(len(I['institutions']))

3409
1817
525


In [5]:
import pickle
B = pickle.load(open('DyHANE/2019-2021/B_1024.pkl', 'rb'))
print(len(B['authors']))
print(len(B['papers']))
print(len(B['institutions']))

74
377
61
